# purple-sectors · 01 · The pace model
One model for every qualifying push lap since 2023: **pace = car + driver + fit**.

1. Fetch the lap files that are missing (2023–2025 are new).
2. Check the design: how many laps, cars, drivers, and whether every driver is linked to every other through shared cars.
3. Prior check, then recovery on a simulated history (does the model find a known truth?).
4. Fit the real laps, run the quality gate, export `site/data/pace.json`, download the results.

Runtime: about 5 min to install CmdStan, then roughly 20–40 min per fit on a Colab CPU.

In [ ]:
# 1 · Drag the newest purple-sectors zip into the Files panel first, then run
import glob, os, zipfile
os.chdir('/content')
z = sorted(glob.glob('/content/purple-sectors*.zip'), key=os.path.getmtime)
assert z, 'No purple-sectors zip in /content. Drag it into the Files panel and wait for the upload.'
zipfile.ZipFile(z[-1]).extractall('/content')
os.chdir('/content/purple-sectors')
print('using', z[-1])

In [ ]:
# 2 · CmdStan (about 5 min, once per Colab session)
%pip install -q cmdstanpy scipy
import cmdstanpy
cmdstanpy.install_cmdstan(progress=True)

In [ ]:
# 3 · Lap files: fetch the seasons that are missing (OpenF1 starts in 2023)
import os, subprocess, sys
os.chdir('/content/purple-sectors'); sys.path.insert(0, '/content/purple-sectors/src')
os.environ['PYTHONPATH'] = 'src'
YEARS = [2023, 2024, 2025, 2026]
for y in YEARS:
    if not os.path.exists(f'data/raw/quali_laps_{y}.csv'):
        print('fetching', y); subprocess.run([sys.executable, '-m', 'purple_sectors.fetch', '--year', str(y)], check=True)
print({y: sum(1 for _ in open(f'data/raw/quali_laps_{y}.csv')) - 1 for y in YEARS}, 'laps per file')

In [ ]:
# 4 · The design
from purple_sectors.pace import data as D
des = D.design_from_files('data/raw', YEARS, 'data/raw/driver_debuts.json')
net = D.connectivity(des)
print(f'{des.N} push laps · {des.D} drivers · {sum(des.T)} cars · {des.E} events · {des.P} driver-car pairs')
print('seasons', des.seasons, '· teams per season', des.T, '· rounds per season', des.R)
print('all drivers linked through shared cars:', net['connected'])
if not net['connected']:
    print('outside the main group (only comparable with their own teammates):', net['components'][1:])
print('rookies by season:', {s: [d for d, flag in zip(des.drivers, des.rookie[:, i]) if flag] for i, s in enumerate(des.seasons)})

In [ ]:
# 5 · Prior check: does the prior produce seasons that look like F1?
import numpy as np
from purple_sectors.pace import simulate as S
rng = np.random.default_rng(0)
spread_car, gap_mate, worst = [], [], []
for _ in range(200):
    p = S.draw_params(des, rng)
    y = S.simulate_laps(des, p, rng)
    spread_car.append(np.ptp(p['car'][-1, :des.T[-1], des.R[-1] - 1]) * 0.9)
    gap_mate.append(np.abs(np.diff(np.sort(p['skill'][des.present[:, -1] == 1, -1])[:2])).item() * 0.9)
    worst.append(np.percentile(y - S.mu_of(des, p), 99) * 0.9)
print(f'car spread at the last round (s on a 90 s lap): 5-50-95% = {np.percentile(spread_car, [5, 50, 95]).round(2)}')
print(f'99th percentile of time lost on a lap (s): 5-50-95% = {np.percentile(worst, [5, 50, 95]).round(2)}')
real = des.y - np.array([des.y_ref[b - 1] for b in des.base])
print(f'real laps: 99th percentile above the segment median = {np.percentile(real, 99) * 0.9:.2f} s')

In [ ]:
# 6 · Recovery: simulate a history from a known truth on the real design, fit, check it comes back
from purple_sectors.pace import fit as F
os.makedirs('out', exist_ok=True)
rec = F.run(F.parse(['--recovery', '--warmup', '600', '--samples', '600', '--report', 'out/recovery.json']))
r = rec['recovery']
print(f"inside 90% intervals: driver skill {r['skill']:.0%} · car {r['car']:.0%} · fit {r['fit']:.0%} · hyperparameters {r['hypers_inside']:.0%}")
print('want 80-95% on each; sampler status:', rec['status'])

In [ ]:
# 7 · The real fit (run only if recovery looks right)
rep = F.run(F.parse(['--out', 'out/posterior_pace.npz', '--report', 'out/fit.json', '--retry']))
print(rep['status'], rep['diagnostics'])
print({k: v for k, v in rep['medians'].items()})

In [ ]:
# 8 · Export for the site and look at the headline numbers
import json
from pathlib import Path
from purple_sectors.pace import export as X
X.main(Path('out/posterior_pace.npz'), Path(f'data/raw/quali_laps_{YEARS[-1]}.csv'), Path('site/data'), Path('site/data/snapshots'))
soft = json.loads(Path('site/data/soft.json').read_text()); pace = json.loads(Path('site/data/pace.json').read_text())
print(f"{soft['season']} after {soft['event']} (round {soft['round']}), {soft['laps']} laps this season, gate: {soft['status']}")
print('\nCars, driver taken out (s against the average car, 90% range):')
for c in soft['cars']: print(f"  {c['team']:14} {c['med']:+.2f}  [{c['lo']:+.2f}, {c['hi']:+.2f}]  P(fastest) {c['p_fast']:.2f}")
print('\nDrivers, car taken out (s against the average driver; rank 10-90%):')
for d in soft['drivers']: print(f"  {d['code']} {d['team']:14} {d['skill'][1]:+.2f}  rank {d['rank']:2} ({d['rank_lo']}-{d['rank_hi']})  {'moved teams' if d['moved_teams'] else 'one team'}{'  rookie' if d['rookie'] else ''}")
print('\nTeammates (gap in s, P(first named ahead)):')
for m in soft['pairs']: print(f"  {m['team']:14} {m['a']['code']} vs {m['b']['code']}: {abs(m['gap']):.2f}  P {m['p']:.2f}")
g = pace['grid']; i = 0
print(f"\n{g['drivers'][i]} in every car (median grid position):", dict(zip(g['cars'], [c['pos'][1] for c in g['cells'][i]])))

In [ ]:
# 9 · Download the posterior, the reports and the site data
import glob
from google.colab import files
with zipfile.ZipFile('pace_outputs.zip', 'w') as zf:
    for f in ['out/posterior_pace.npz', 'out/fit.json', 'out/recovery.json'] + glob.glob('site/data/**/*.json', recursive=True):
        if os.path.exists(f): zf.write(f)
    for y in YEARS:
        zf.write(f'data/raw/quali_laps_{y}.csv')  # commit the new seasons with the repo
files.download('pace_outputs.zip')